In [1]:
import os
import rasterio
import numpy as np

# Define settings
directory_path = '/content/drive/MyDrive/CCRI/malaria'
nodata_val = -9999  # <--- This is your required final value

# Get files
all_files = [f for f in os.listdir(directory_path) if f.lower().endswith(('.tif', '.tiff'))]
tif_files = sorted(all_files)[-10:]

data_list = []

for file in tif_files:
    file_path = os.path.join(directory_path, file)
    with rasterio.open(file_path) as src:
        # Read as float so we can use NaNs
        data = src.read(1).astype(np.float32)

        # Convert the input -9999 to NaN so math works correctly
        data[data == nodata_val] = np.nan
        data_list.append(data)

# Stack and average ignoring NaNs
stacked_data = np.stack(data_list, axis=0)
average_data = np.nanmean(stacked_data, axis=0)

# --- CRITICAL STEP: FORCE FINAL NODATA TO -9999 ---

# 1. Replace all remaining NaNs (where no data existed) with -9999
average_data = np.nan_to_num(average_data, nan=nodata_val)

# 2. Get profile and update it
with rasterio.open(os.path.join(directory_path, tif_files[0])) as src:
    profile = src.profile

profile.update(
    count=1,
    dtype=rasterio.float32,
    nodata=nodata_val  # <--- Writes metadata saying "-9999 is NoData"
)

# Save
output_path = os.path.join(directory_path, 'Global_Pf_average_2013_2022.tif')
with rasterio.open(output_path, 'w', **profile) as dst:
    dst.write(average_data, 1)

print(f"Saved average raster to {output_path}")
print(f"NoData value set to: {nodata_val}")

/tmp/ipython-input-1170733467.py:27: RuntimeWarning: Mean of empty slice
  average_data = np.nanmean(stacked_data, axis=0)


Saved average raster to /content/drive/MyDrive/CCRI/malaria/Global_Pf_average_2013_2022.tif
NoData value set to: -9999


In [ ]:
!pip install rasterio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 44.4 MB/s eta 0:00:00


In [2]:
import os
import rasterio
import numpy as np

# Define the directory path
directory_path = '/content/drive/MyDrive/CCRI/malaria'
nodata_val = -9999  # <--- Define NoData value

# Get a list of all TIFF files (filtering for '_Pv_')
all_files = [f for f in os.listdir(directory_path)
             if f.lower().endswith(('.tif', '.tiff')) and '_Pv_' in f]

# Select the last 10 files (alphabetical order)
tif_files = sorted(all_files)[-10:]

# Initialize a list to store arrays
data_list = []

for file in tif_files:
    file_path = os.path.join(directory_path, file)
    with rasterio.open(file_path) as src:
        data = src.read(1).astype(np.float32)  # Read first band, ensure float32

        # 1. MASK INPUT: Convert -9999 to NaN so math works correctly
        data[data == nodata_val] = np.nan

        data_list.append(data)

# Stack along a new axis
stacked_data = np.stack(data_list, axis=0)  # Shape: (10, height, width)

# 2. CALCULATE MEAN: Use nanmean to ignore the NaNs we created
average_data = np.nanmean(stacked_data, axis=0)

# 3. RESTORE NODATA: Fill NaNs back to -9999 for the output file
average_data = np.nan_to_num(average_data, nan=nodata_val)

# Read metadata from one file
with rasterio.open(os.path.join(directory_path, tif_files[0])) as src:
    profile = src.profile

# Update profile for output
profile.update(
    count=1,
    dtype=rasterio.float32,
    nodata=nodata_val  # <--- Important: Tell GIS software -9999 is transparent
)

# Save the averaged result as a GeoTIFF
output_path = os.path.join(directory_path, 'Global_Pv_average_2013_2022.tif')
with rasterio.open(output_path, 'w', **profile) as dst:
    dst.write(average_data, 1)  # Write to first band

print(f"Saved average raster: {output_path}")
print(f"NoData value set to: {nodata_val}")

/tmp/ipython-input-3052925343.py:33: RuntimeWarning: Mean of empty slice
  average_data = np.nanmean(stacked_data, axis=0)


Saved average raster: /content/drive/MyDrive/CCRI/malaria/Global_Pv_average_2013_2022.tif
NoData value set to: -9999
